In [0]:
# Importación de librerías necesarias para generación y manipulación de datos
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
import random
from datetime import datetime, timedelta

In [0]:
# Generación de 2000 registros simulados de ventas
productos = [101,102,103,104,105,106,107,108,109,110]
tiendas = [1,2,3,4,5]
clientes = list(range(1001,1101))

ventas = []

fecha_inicio = datetime(2026, 1, 1)

for i in range(1,2001):

    fecha = fecha_inicio + timedelta(days=random.randint(0,120))

    venta = (
        i,
        fecha.strftime("%Y-%m-%d"),
        random.choice(productos),
        random.choice(tiendas),
        random.choice(clientes),
        random.randint(1,10),
        random.randint(500,10000)
    )

    ventas.append(venta)

In [0]:
# Creación del DataFrame de ventas con estructura definida
columnas = [
    "id_venta",
    "fecha_venta",
    "id_producto",
    "id_tienda",
    "id_cliente",
    "cantidad",
    "precio_unitario"
]

df_ventas = spark.createDataFrame(ventas, columnas)

In [0]:
# Visualización inicial del dataset de ventas
display(df_ventas)

In [0]:
# Incorporación de registros erróneos para pruebas de calidad de datos
errores = [
    (2001, "2026-05-01", 999, 1, 1001, 2, 1500),
    (2002, "2026-05-02", 101, 2, 1002, -3, 2000),
    (2002, "2026-05-02", 101, 2, 1002, -3, 2000)
]

df_errores = spark.createDataFrame(errores, columnas)

df_ventas = df_ventas.union(df_errores)

In [0]:
# Visualización del dataset de ventas con errores intencionales
display(df_ventas)

In [0]:
# Creación y visualización del dataset de productos
productos_data = [
    (101, "Notebook", "Tecnología", "Lenovo"),
    (102, "Mouse", "Tecnología", "Logitech"),
    (103, "Teclado", "Tecnología", "Redragon"),
    (104, "Monitor", "Tecnología", "Samsung"),
    (105, "Auriculares", "Tecnología", "Sony"),
    (106, "Celular", "Tecnología", "Motorola"),
    (107, "Tablet", "Tecnología", "Apple"),
    (108, "Impresora", "Tecnología", "HP"),
    (109, "Webcam", "Tecnología", "Logitech"),
    (110, "Parlante", "Tecnología", "JBL")
]

columnas_productos = [
    "id_producto",
    "nombre_producto",
    "categoria",
    "marca"
]

df_productos = spark.createDataFrame(productos_data, columnas_productos)

display(df_productos)

id_producto,nombre_producto,categoria,marca
101,Notebook,Tecnología,Lenovo
102,Mouse,Tecnología,Logitech
103,Teclado,Tecnología,Redragon
104,Monitor,Tecnología,Samsung
105,Auriculares,Tecnología,Sony
106,Celular,Tecnología,Motorola
107,Tablet,Tecnología,Apple
108,Impresora,Tecnología,HP
109,Webcam,Tecnología,Logitech
110,Parlante,Tecnología,JBL


In [0]:
# Creación y visualización del dataset de tiendas
tiendas_data = [
    (1, "Sucursal Centro", "Buenos Aires", "Buenos Aires"),
    (2, "Sucursal Norte", "Córdoba", "Córdoba"),
    (3, "Sucursal Sur", "Rosario", "Santa Fe"),
    (4, "Sucursal Oeste", "Mendoza", "Mendoza"),
    (5, "Sucursal Este", "La Plata", "Buenos Aires")
]

columnas_tiendas = [
    "id_tienda",
    "nombre_tienda",
    "ciudad",
    "provincia"
]

df_tiendas = spark.createDataFrame(tiendas_data, columnas_tiendas)

display(df_tiendas)

id_tienda,nombre_tienda,ciudad,provincia
1,Sucursal Centro,Buenos Aires,Buenos Aires
2,Sucursal Norte,Córdoba,Córdoba
3,Sucursal Sur,Rosario,Santa Fe
4,Sucursal Oeste,Mendoza,Mendoza
5,Sucursal Este,La Plata,Buenos Aires


In [0]:
# Generación y visualización del dataset de clientes
clientes_data = []

for i in range(1001,1101):

    genero = random.choice(["Masculino", "Femenino"])

    edad = random.randint(18,70)

    clientes_data.append(
        (i, f"Cliente_{i}", genero, edad)
    )

columnas_clientes = [
    "id_cliente",
    "nombre_cliente",
    "genero",
    "edad"
]

df_clientes = spark.createDataFrame(clientes_data, columnas_clientes)

display(df_clientes)

id_cliente,nombre_cliente,genero,edad
1001,Cliente_1001,Femenino,26
1002,Cliente_1002,Masculino,51
1003,Cliente_1003,Masculino,60
1004,Cliente_1004,Femenino,32
1005,Cliente_1005,Femenino,67
1006,Cliente_1006,Femenino,37
1007,Cliente_1007,Masculino,51
1008,Cliente_1008,Femenino,57
1009,Cliente_1009,Masculino,19
1010,Cliente_1010,Femenino,29


In [0]:
# Visualización de la estructura y tipos de datos del dataset de ventas
df_ventas.printSchema()

root
 |-- id_venta: long (nullable = true)
 |-- fecha_venta: string (nullable = true)
 |-- id_producto: long (nullable = true)
 |-- id_tienda: long (nullable = true)
 |-- id_cliente: long (nullable = true)
 |-- cantidad: long (nullable = true)
 |-- precio_unitario: long (nullable = true)



In [0]:
# Validación de cantidad de registros por dataset
print("Ventas:", df_ventas.count())
print("Productos:", df_productos.count())
print("Tiendas:", df_tiendas.count())
print("Clientes:", df_clientes.count())

Ventas: 2003
Productos: 10
Tiendas: 5
Clientes: 100


In [0]:
# Verificación de valores nulos en el dataset de ventas

df_ventas.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_ventas.columns
]).show()

+--------+-----------+-----------+---------+----------+--------+---------------+
|id_venta|fecha_venta|id_producto|id_tienda|id_cliente|cantidad|precio_unitario|
+--------+-----------+-----------+---------+----------+--------+---------------+
|       0|          0|          0|        0|         0|       0|              0|
+--------+-----------+-----------+---------+----------+--------+---------------+



In [0]:
# Verificación de registros duplicados por id_venta

df_ventas.groupBy("id_venta") \
    .count() \
    .filter(col("count") > 1) \
    .show()

+--------+-----+
|id_venta|count|
+--------+-----+
|    2002|    2|
+--------+-----+



In [0]:
# Verificación de productos inexistentes en ventas

df_ventas.join(
    df_productos,
    on="id_producto",
    how="left"
).filter(
    col("nombre_producto").isNull()
).show()

+-----------+--------+-----------+---------+----------+--------+---------------+---------------+---------+-----+
|id_producto|id_venta|fecha_venta|id_tienda|id_cliente|cantidad|precio_unitario|nombre_producto|categoria|marca|
+-----------+--------+-----------+---------+----------+--------+---------------+---------------+---------+-----+
|        999|    2001| 2026-05-01|        1|      1001|       2|           1500|           NULL|     NULL| NULL|
+-----------+--------+-----------+---------+----------+--------+---------------+---------------+---------+-----+



# Diseño del esquema relacional

Relaciones definidas:

- ventas.id_producto -> productos.id_producto
- ventas.id_tienda -> tiendas.id_tienda
- ventas.id_cliente -> clientes.id_cliente

Granularidad:
- Una fila representa una transacción de venta.

Tablas dimensionales:
- productos
- tiendas
- clientes

Tabla transaccional:
- ventas

In [0]:
# Visualización de sistemas de archivos disponibles

dbutils.fs.ls("/")

[FileInfo(path='dbfs:/Volumes/', name='Volumes/', size=0, modificationTime=0),
 FileInfo(path='dbfs:/Workspace/', name='Workspace/', size=0, modificationTime=0),
 FileInfo(path='dbfs:/databricks/', name='databricks/', size=0, modificationTime=0),
 FileInfo(path='dbfs:/databricks-datasets/', name='databricks-datasets/', size=0, modificationTime=0)]

In [0]:
# Configuración de credenciales para acceso al Blob Storage

spark.conf.set(
    "fs.azure.account.key.TU_STORAGE.blob.core.windows.net",
    "TU_KEY_AQUI"
)

In [0]:
# Definición de ruta RAW en Azure Blob Storage

ruta_raw = "wasbs://raw@TU_STORAGE.blob.core.windows.net/"

In [0]:
# Escritura del dataset de ventas en contenedor RAW

df_ventas.write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv(ruta_raw + "ventas")

In [0]:
# Escritura de datasets dimensionales en contenedor RAW

df_productos.write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv(ruta_raw + "productos")

df_tiendas.write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv(ruta_raw + "tiendas")

df_clientes.write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv(ruta_raw + "clientes")

In [0]:
# Verificación de carpetas guardadas en el contenedor RAW

dbutils.fs.ls(ruta_raw)

[FileInfo(path='wasbs://raw@diplomaturastorage2026.blob.core.windows.net/clientes/', name='clientes/', size=0, modificationTime=1778364394000),
 FileInfo(path='wasbs://raw@diplomaturastorage2026.blob.core.windows.net/productos/', name='productos/', size=0, modificationTime=1778364392000),
 FileInfo(path='wasbs://raw@diplomaturastorage2026.blob.core.windows.net/tiendas/', name='tiendas/', size=0, modificationTime=1778364393000),
 FileInfo(path='wasbs://raw@diplomaturastorage2026.blob.core.windows.net/ventas/', name='ventas/', size=0, modificationTime=1778364351000)]

In [0]:
# Lectura de dataset de ventas desde Azure Blob Storage

df_test = spark.read \
    .option("header", "true") \
    .csv(ruta_raw + "ventas")

display(df_test)